In [1]:
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import sys
from pathlib import Path
import sklearn
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import root_mean_squared_error

project_root = Path.cwd().parent
sys.path.append(str(project_root))

In [2]:
from src.data import load_wr_weekly_stats

receiving_stats = load_wr_weekly_stats([2021, 2022, 2023, 2024, 2025])
receiving_stats

player_display_name,season,season_type,week,team,opponent_team,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles_lost,receiving_2pt_conversions,special_teams_tds,rushing_yards,rushing_tds,rushing_fumbles_lost,rushing_2pt_conversions,fantasy_points_ppr
str,i32,str,i32,str,str,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,f64
"""Danny Amendola""",2021,"""REG""",1,"""HOU""","""JAX""",5,5,34,1,0,0,0,0,0,0,0,14.4
"""DeSean Jackson""",2021,"""REG""",1,"""LA""","""CHI""",2,2,21,0,0,0,0,0,0,0,0,4.1
"""Emmanuel Sanders""",2021,"""REG""",1,"""BUF""","""PIT""",4,8,52,0,0,0,0,0,0,0,0,9.2
"""Andre Roberts""",2021,"""REG""",1,"""HOU""","""JAX""",0,0,0,0,0,0,0,0,0,0,0,0.0
"""Antonio Brown""",2021,"""REG""",1,"""TB""","""DAL""",5,7,121,1,0,0,0,6,0,0,0,23.7
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Chimere Dike""",2025,"""REG""",18,"""TEN""","""JAX""",3,4,27,0,0,0,0,0,0,0,0,5.7
"""Tre Harris""",2025,"""REG""",18,"""LAC""","""DEN""",2,6,28,0,0,0,0,0,0,0,0,4.8
"""Jack Bech""",2025,"""REG""",18,"""LV""","""KC""",0,2,0,0,0,0,0,0,0,0,0,0.0


In [3]:
from src.scoring import calculate_ppr

receiving_stats_ppr = calculate_ppr(receiving_stats)
receiving_stats_ppr

player_display_name,season,season_type,week,team,opponent_team,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles_lost,receiving_2pt_conversions,special_teams_tds,rushing_yards,rushing_tds,rushing_fumbles_lost,rushing_2pt_conversions,fantasy_points_ppr,calculated_ppr
str,i32,str,i32,str,str,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,f64,f64
"""Danny Amendola""",2021,"""REG""",1,"""HOU""","""JAX""",5,5,34,1,0,0,0,0,0,0,0,14.4,14.4
"""DeSean Jackson""",2021,"""REG""",1,"""LA""","""CHI""",2,2,21,0,0,0,0,0,0,0,0,4.1,4.1
"""Emmanuel Sanders""",2021,"""REG""",1,"""BUF""","""PIT""",4,8,52,0,0,0,0,0,0,0,0,9.2,9.2
"""Andre Roberts""",2021,"""REG""",1,"""HOU""","""JAX""",0,0,0,0,0,0,0,0,0,0,0,0.0,0.0
"""Antonio Brown""",2021,"""REG""",1,"""TB""","""DAL""",5,7,121,1,0,0,0,6,0,0,0,23.7,23.7
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Chimere Dike""",2025,"""REG""",18,"""TEN""","""JAX""",3,4,27,0,0,0,0,0,0,0,0,5.7,5.7
"""Tre Harris""",2025,"""REG""",18,"""LAC""","""DEN""",2,6,28,0,0,0,0,0,0,0,0,4.8,4.8
"""Jack Bech""",2025,"""REG""",18,"""LV""","""KC""",0,2,0,0,0,0,0,0,0,0,0,0.0,0.0


In [4]:
from src.features import create_features
from src.features import create_defensive_features
from src.modeling import FEATURE_COLUMNS, TARGET_COLUMN, make_train_test_data, train_linear_regression, evaluate_predictions

player_prediction_features = create_features(receiving_stats_ppr)
player_prediction_features = create_defensive_features(player_prediction_features)

player_prediction_features

player_display_name,season,season_type,week,team,opponent_team,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles_lost,receiving_2pt_conversions,special_teams_tds,rushing_yards,rushing_tds,rushing_fumbles_lost,rushing_2pt_conversions,fantasy_points_ppr,calculated_ppr,prev_rolling_3_ppr,prev_rolling_3_targets,prev_rolling_3_receptions,prev_rolling_3_receiving_yards,prev_season_avg_ppr,prev_defense_wr_ppr_allowed_avg
str,i32,str,i32,str,str,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,f64,f64,f64,f64,f64,f64,f64,f64
"""A.J. Brown""",2021,"""REG""",1,"""TEN""","""ARI""",4,8,49,1,0,0,0,0,0,0,0,14.9,14.9,null,null,null,null,null,null
"""A.J. Brown""",2021,"""REG""",2,"""TEN""","""SEA""",3,9,43,0,0,0,0,0,0,0,0,7.3,7.3,null,null,null,null,14.9,34.2
"""A.J. Brown""",2021,"""REG""",3,"""TEN""","""IND""",0,2,0,0,0,0,0,3,0,0,0,0.3,0.3,null,null,null,null,11.1,48.75
"""A.J. Brown""",2021,"""REG""",5,"""TEN""","""JAX""",3,6,38,0,0,0,0,0,0,0,0,6.8,6.8,7.5,6.33,2.33,30.67,7.5,40.88
"""A.J. Brown""",2021,"""REG""",6,"""TEN""","""BUF""",7,9,91,0,0,0,0,0,0,0,0,16.1,16.1,4.8,5.67,2.0,27.0,7.32,26.62
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Zay Jones""",2025,"""REG""",3,"""ARI""","""SF""",2,3,25,0,0,0,0,0,0,0,0,4.5,4.5,4.3,2.33,2.0,23.0,1.4,30.7
"""Zay Jones""",2025,"""REG""",5,"""ARI""","""TEN""",2,3,8,0,0,0,0,0,0,0,0,2.8,2.8,4.83,3.0,2.33,25.0,2.95,38.7
"""Zay Jones""",2025,"""REG""",6,"""ARI""","""IND""",5,8,79,0,0,0,0,0,0,0,0,12.9,12.9,2.9,2.33,1.67,12.33,2.9,38.02


In [5]:
with pl.Config(tbl_rows=-1):
    print(player_prediction_features.filter(pl.col("player_display_name") == 'CeeDee Lamb').head(20))
# player_prediction_features.filter(
#     pl.col("player_display_name") == "A.J. Brown"
# ).select([
#     "player_display_name",
#     "season",
#     "week",
#     "calculated_ppr",
#     "prev_rolling_3_ppr",
#     "targets",
#     "prev_rolling_3_targets",
#     "receptions",
#     "prev_rolling_3_receptions",
#     "receiving_yards",
#     "prev_rolling_3_receiving_yards"
# ]).sort(["season", "week"])

shape: (20, 25)
┌─────────────┬────────┬────────────┬──────┬───┬────────────┬────────────┬────────────┬────────────┐
│ player_disp ┆ season ┆ season_typ ┆ week ┆ … ┆ prev_rolli ┆ prev_rolli ┆ prev_seaso ┆ prev_defen │
│ lay_name    ┆ ---    ┆ e          ┆ ---  ┆   ┆ ng_3_recep ┆ ng_3_recei ┆ n_avg_ppr  ┆ se_wr_ppr_ │
│ ---         ┆ i32    ┆ ---        ┆ i32  ┆   ┆ tions      ┆ ving_yards ┆ ---        ┆ allowed_av │
│ str         ┆        ┆ str        ┆      ┆   ┆ ---        ┆ ---        ┆ f64        ┆ …          │
│             ┆        ┆            ┆      ┆   ┆ f64        ┆ f64        ┆            ┆ ---        │
│             ┆        ┆            ┆      ┆   ┆            ┆            ┆            ┆ f64        │
╞═════════════╪════════╪════════════╪══════╪═══╪════════════╪════════════╪════════════╪════════════╡
│ CeeDee Lamb ┆ 2021   ┆ REG        ┆ 1    ┆ … ┆ null       ┆ null       ┆ null       ┆ null       │
│ CeeDee Lamb ┆ 2021   ┆ REG        ┆ 2    ┆ … ┆ null       ┆ null       ┆ 

In [6]:
player_prediction_features = (
    player_prediction_features
    .sort(["player_display_name", "season", "week"])
    .with_columns(
        (
            pl.col("calculated_ppr").shift(1).cum_sum().over(["player_display_name", "season"])
            /
            pl.col("calculated_ppr").shift(1).cum_count().over(["player_display_name", "season"])
        )
        .round(2)
        .alias("prev_season_avg_ppr")
    )
)

player_prediction_features

player_display_name,season,season_type,week,team,opponent_team,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles_lost,receiving_2pt_conversions,special_teams_tds,rushing_yards,rushing_tds,rushing_fumbles_lost,rushing_2pt_conversions,fantasy_points_ppr,calculated_ppr,prev_rolling_3_ppr,prev_rolling_3_targets,prev_rolling_3_receptions,prev_rolling_3_receiving_yards,prev_season_avg_ppr,prev_defense_wr_ppr_allowed_avg
str,i32,str,i32,str,str,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,f64,f64,f64,f64,f64,f64,f64,f64
"""A.J. Brown""",2021,"""REG""",1,"""TEN""","""ARI""",4,8,49,1,0,0,0,0,0,0,0,14.9,14.9,null,null,null,null,null,null
"""A.J. Brown""",2021,"""REG""",2,"""TEN""","""SEA""",3,9,43,0,0,0,0,0,0,0,0,7.3,7.3,null,null,null,null,14.9,34.2
"""A.J. Brown""",2021,"""REG""",3,"""TEN""","""IND""",0,2,0,0,0,0,0,3,0,0,0,0.3,0.3,null,null,null,null,11.1,48.75
"""A.J. Brown""",2021,"""REG""",5,"""TEN""","""JAX""",3,6,38,0,0,0,0,0,0,0,0,6.8,6.8,7.5,6.33,2.33,30.67,7.5,40.88
"""A.J. Brown""",2021,"""REG""",6,"""TEN""","""BUF""",7,9,91,0,0,0,0,0,0,0,0,16.1,16.1,4.8,5.67,2.0,27.0,7.32,26.62
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Zay Jones""",2025,"""REG""",3,"""ARI""","""SF""",2,3,25,0,0,0,0,0,0,0,0,4.5,4.5,4.3,2.33,2.0,23.0,1.4,30.7
"""Zay Jones""",2025,"""REG""",5,"""ARI""","""TEN""",2,3,8,0,0,0,0,0,0,0,0,2.8,2.8,4.83,3.0,2.33,25.0,2.95,38.7
"""Zay Jones""",2025,"""REG""",6,"""ARI""","""IND""",5,8,79,0,0,0,0,0,0,0,0,12.9,12.9,2.9,2.33,1.67,12.33,2.9,38.02


In [7]:
player_prediction_features.filter(
    pl.col("player_display_name") == "A.J. Brown"
).select([
    "player_display_name",
    "season",
    "week",
    "calculated_ppr",
    "prev_season_avg_ppr"
]).sort(["season", "week"])

player_display_name,season,week,calculated_ppr,prev_season_avg_ppr
str,i32,i32,f64,f64
"""A.J. Brown""",2021,1,14.9,null
"""A.J. Brown""",2021,2,7.3,14.9
"""A.J. Brown""",2021,3,0.3,11.1
"""A.J. Brown""",2021,5,6.8,7.5
"""A.J. Brown""",2021,6,16.1,7.32
…,…,…,…,…
"""A.J. Brown""",2025,13,35.2,12.67
"""A.J. Brown""",2025,14,16.0,14.72
"""A.J. Brown""",2025,15,12.1,14.82


In [8]:
baseline_prediction_clean = player_prediction_features.filter(
    pl.col("prev_rolling_3_ppr").is_not_null()
)

baseline_prediction_clean

player_display_name,season,season_type,week,team,opponent_team,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles_lost,receiving_2pt_conversions,special_teams_tds,rushing_yards,rushing_tds,rushing_fumbles_lost,rushing_2pt_conversions,fantasy_points_ppr,calculated_ppr,prev_rolling_3_ppr,prev_rolling_3_targets,prev_rolling_3_receptions,prev_rolling_3_receiving_yards,prev_season_avg_ppr,prev_defense_wr_ppr_allowed_avg
str,i32,str,i32,str,str,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,f64,f64,f64,f64,f64,f64,f64,f64
"""A.J. Brown""",2021,"""REG""",5,"""TEN""","""JAX""",3,6,38,0,0,0,0,0,0,0,0,6.8,6.8,7.5,6.33,2.33,30.67,7.5,40.88
"""A.J. Brown""",2021,"""REG""",6,"""TEN""","""BUF""",7,9,91,0,0,0,0,0,0,0,0,16.1,16.1,4.8,5.67,2.0,27.0,7.32,26.62
"""A.J. Brown""",2021,"""REG""",7,"""TEN""","""KC""",8,9,133,1,0,0,0,0,0,0,0,27.3,27.3,7.73,5.67,3.33,43.0,9.08,33.08
"""A.J. Brown""",2021,"""REG""",8,"""TEN""","""IND""",10,11,155,1,0,0,0,0,0,0,0,31.5,31.5,16.73,8.0,6.0,87.33,12.12,38.4
"""A.J. Brown""",2021,"""REG""",9,"""TEN""","""LA""",5,11,42,0,0,0,0,0,0,0,0,9.2,9.2,24.97,9.67,8.33,126.33,14.89,36.88
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Zay Jones""",2025,"""REG""",3,"""ARI""","""SF""",2,3,25,0,0,0,0,0,0,0,0,4.5,4.5,4.3,2.33,2.0,23.0,1.4,30.7
"""Zay Jones""",2025,"""REG""",5,"""ARI""","""TEN""",2,3,8,0,0,0,0,0,0,0,0,2.8,2.8,4.83,3.0,2.33,25.0,2.95,38.7
"""Zay Jones""",2025,"""REG""",6,"""ARI""","""IND""",5,8,79,0,0,0,0,0,0,0,0,12.9,12.9,2.9,2.33,1.67,12.33,2.9,38.02


In [9]:
dataset = (
    baseline_prediction_clean.select([
        'player_display_name',
        'season',
        'week',
        'team',
        'opponent_team',
        'prev_rolling_3_ppr',
        'prev_rolling_3_targets',
        'prev_rolling_3_receptions',
        'prev_rolling_3_receiving_yards',
        'prev_season_avg_ppr',
        'prev_defense_wr_ppr_allowed_avg',
        'calculated_ppr',
    ])
)

dataset = dataset.filter(
    (pl.col("prev_rolling_3_ppr").is_not_null()) & 
    (pl.col("prev_rolling_3_targets").is_not_null()) & 
    (pl.col("prev_rolling_3_receptions").is_not_null()) & 
    (pl.col("prev_rolling_3_receiving_yards").is_not_null()) & 
    (pl.col("prev_season_avg_ppr").is_not_null()) & 
    (pl.col("prev_defense_wr_ppr_allowed_avg").is_not_null()) & 
    (pl.col("calculated_ppr").is_not_null())
)

dataset

player_display_name,season,week,team,opponent_team,prev_rolling_3_ppr,prev_rolling_3_targets,prev_rolling_3_receptions,prev_rolling_3_receiving_yards,prev_season_avg_ppr,prev_defense_wr_ppr_allowed_avg,calculated_ppr
str,i32,i32,str,str,f64,f64,f64,f64,f64,f64,f64
"""A.J. Brown""",2021,5,"""TEN""","""JAX""",7.5,6.33,2.33,30.67,7.5,40.88,6.8
"""A.J. Brown""",2021,6,"""TEN""","""BUF""",4.8,5.67,2.0,27.0,7.32,26.62,16.1
"""A.J. Brown""",2021,7,"""TEN""","""KC""",7.73,5.67,3.33,43.0,9.08,33.08,27.3
"""A.J. Brown""",2021,8,"""TEN""","""IND""",16.73,8.0,6.0,87.33,12.12,38.4,31.5
"""A.J. Brown""",2021,9,"""TEN""","""LA""",24.97,9.67,8.33,126.33,14.89,36.88,9.2
…,…,…,…,…,…,…,…,…,…,…,…
"""Zay Jones""",2025,3,"""ARI""","""SF""",4.3,2.33,2.0,23.0,1.4,30.7,4.5
"""Zay Jones""",2025,5,"""ARI""","""TEN""",4.83,3.0,2.33,25.0,2.95,38.7,2.8
"""Zay Jones""",2025,6,"""ARI""","""IND""",2.9,2.33,1.67,12.33,2.9,38.02,12.9


In [10]:
test_2025 = dataset.filter(pl.col("season") == 2025)

rolling_mae, rolling_rmse = evaluate_predictions(
    test_2025["calculated_ppr"].to_numpy(),
    test_2025["prev_rolling_3_ppr"].to_numpy(),
)
print(f"Rolling 3-game baseline MAE: {rolling_mae}")
print(f"Rolling 3-game baseline RMSE: {rolling_rmse}")

Rolling 3-game baseline MAE: 4.56
Rolling 3-game baseline RMSE: 6.41


In [11]:
season_avg_mae, season_avg_rmse = evaluate_predictions(
    test_2025["calculated_ppr"].to_numpy(),
    test_2025["prev_season_avg_ppr"].to_numpy(),
)
print(f"Season-to-date baseline MAE: {season_avg_mae}")
print(f"Season-to-date baseline RMSE: {season_avg_rmse}")

Season-to-date baseline MAE: 4.45
Season-to-date baseline RMSE: 6.27


In [12]:
feature_columns = FEATURE_COLUMNS
feature_columns


['prev_rolling_3_ppr',
 'prev_rolling_3_targets',
 'prev_rolling_3_receptions',
 'prev_rolling_3_receiving_yards',
 'prev_season_avg_ppr',
 'prev_defense_wr_ppr_allowed_avg']

In [13]:
X_train, X_test, y_train, y_test = make_train_test_data(
    dataset,
    FEATURE_COLUMNS,
    TARGET_COLUMN,
)

model, mae, rmse = train_linear_regression(dataset)
y_pred = model.predict(X_test)

print(f"Mean Absolute Error: {mae}")
print(f"Root Mean Squared Error: {rmse}")


Mean Absolute Error: 4.39
Root Mean Squared Error: 6.0


In [14]:
random_forest = RandomForestRegressor(n_estimators=200, random_state=42, min_samples_leaf=5)

random_forest.fit(X_train, y_train)

rf_y_pred = random_forest.predict(X_test)
rf_mae = mean_absolute_error(y_test, rf_y_pred)
rf_rmse = root_mean_squared_error(y_test, rf_y_pred)
print(f"Random Forest Mean Absolute Error: {rf_mae}")
print(f"Random Forest Root Mean Squared Error: {rf_rmse}")

Random Forest Mean Absolute Error: 4.458324341378723
Random Forest Root Mean Squared Error: 6.120431530180162


In [15]:
test_results = (
    dataset
    .filter(pl.col("season") == 2025)
    .with_columns(
        pl.Series("linear_regression_prediction", y_pred),
        pl.Series("random_forest_prediction", rf_y_pred)
    )
)

test_results = test_results.with_columns(
    (pl.col("linear_regression_prediction") - pl.col("calculated_ppr"))
    .abs()
    .round(2)
    .alias("linear_regression_absolute_error"),

    (pl.col("prev_rolling_3_ppr") - pl.col("calculated_ppr"))
    .abs()
    .round(2)
    .alias("rolling_baseline_absolute_error"),

    (pl.col("prev_season_avg_ppr") - pl.col("calculated_ppr"))
    .abs()
    .round(2)
    .alias("season_baseline_absolute_error")
)

test_results.select([
    "player_display_name",
    "season",
    "week",
    "calculated_ppr",
    "linear_regression_prediction",
    "random_forest_prediction",
    "linear_regression_absolute_error",
    "rolling_baseline_absolute_error",
    "season_baseline_absolute_error",
]).sort("linear_regression_absolute_error", descending=True).head(20)

player_display_name,season,week,calculated_ppr,linear_regression_prediction,random_forest_prediction,linear_regression_absolute_error,rolling_baseline_absolute_error,season_baseline_absolute_error
str,i32,i32,f64,f64,f64,f64,f64,f64
"""Tre Tucker""",2025,3,40.9,8.378128,6.962419,32.52,33.0,31.75
"""Amon-Ra St. Brown""",2025,2,39.2,10.467622,10.256028,28.73,25.13,30.7
"""Puka Nacua""",2025,16,46.5,18.969998,18.932626,27.53,20.9,24.44
"""Amon-Ra St. Brown""",2025,15,41.4,14.792785,13.990254,26.61,26.37,23.44
"""Michael Wilson""",2025,11,33.5,7.479754,7.125559,26.02,25.67,27.82
…,…,…,…,…,…,…,…,…
"""Dontayvion Wicks""",2025,13,28.0,6.394321,6.890154,21.61,23.03,23.41
"""George Pickens""",2025,4,33.4,12.153539,11.610053,21.25,19.53,19.53
"""Alec Pierce""",2025,18,29.2,8.025106,7.798118,21.17,20.13,18.19


In [16]:
test_results = test_results.with_columns(
    pl.Series("random_forest_prediction", rf_y_pred)
)

test_results = test_results.with_columns(
    (pl.col("random_forest_prediction") - pl.col("calculated_ppr"))
    .round(2)
    .alias("random_forest_signed_error"),

    (pl.col("random_forest_prediction") - pl.col("calculated_ppr"))
    .abs()
    .round(2)
    .alias("random_forest_absolute_error"),
)

test_results.select([
    "player_display_name",
    "season",
    "week",
    "calculated_ppr",
    "linear_regression_prediction",
    "linear_regression_absolute_error",
    "random_forest_prediction",
    "random_forest_absolute_error",
    "random_forest_signed_error",
]).sort("random_forest_absolute_error", descending=True).head(20)

player_display_name,season,week,calculated_ppr,linear_regression_prediction,linear_regression_absolute_error,random_forest_prediction,random_forest_absolute_error,random_forest_signed_error
str,i32,i32,f64,f64,f64,f64,f64,f64
"""Tre Tucker""",2025,3,40.9,8.378128,32.52,6.962419,33.94,-33.94
"""Amon-Ra St. Brown""",2025,2,39.2,10.467622,28.73,10.256028,28.94,-28.94
"""Puka Nacua""",2025,16,46.5,18.969998,27.53,18.932626,27.57,-27.57
"""Amon-Ra St. Brown""",2025,15,41.4,14.792785,26.61,13.990254,27.41,-27.41
"""Michael Wilson""",2025,11,33.5,7.479754,26.02,7.125559,26.37,-26.37
…,…,…,…,…,…,…,…,…
"""Drake London""",2025,6,31.8,12.77738,19.02,11.007585,20.79,-20.79
"""Xavier Legette""",2025,7,24.2,5.764764,18.44,3.616403,20.58,-20.58
"""Ryan Flournoy""",2025,14,26.5,4.651428,21.85,6.030206,20.47,-20.47


In [17]:
metrics_comparison = pl.DataFrame({
    "model": [
        "Rolling 3-game baseline",
        "Season-to-date baseline",
        "Linear Regression",
        "Random Forest",
    ],
    "mae": [
        mean_absolute_error(
            test_results["calculated_ppr"],
            test_results["prev_rolling_3_ppr"],
        ),
        mean_absolute_error(
            test_results["calculated_ppr"],
            test_results["prev_season_avg_ppr"],
        ),
        mean_absolute_error(y_test, y_pred),
        mean_absolute_error(y_test, rf_y_pred),
    ],
    "rmse": [
        root_mean_squared_error(
            test_results["calculated_ppr"],
            test_results["prev_rolling_3_ppr"],
        ),
        root_mean_squared_error(
            test_results["calculated_ppr"],
            test_results["prev_season_avg_ppr"],
        ),
        root_mean_squared_error(y_test, y_pred),
        root_mean_squared_error(y_test, rf_y_pred),
    ],
}).with_columns(
    pl.col("mae").round(2),
    pl.col("rmse").round(2),
)

metrics_comparison

model,mae,rmse
str,f64,f64
"""Rolling 3-game baseline""",4.56,6.41
"""Season-to-date baseline""",4.45,6.27
"""Linear Regression""",4.39,6.0
"""Random Forest""",4.46,6.12


Random Forest was tested as a comparison model using the same training and test split as Linear Regression. It performed worse than Linear Regression on the 2025 test season, so Linear Regression remains the preferred current model. This suggests that, with the current feature set, a more flexible model may be fitting noise rather than improving generalization.

In [18]:
coefficients = pl.DataFrame({
    "feature": FEATURE_COLUMNS,
    "coefficient": model.coef_,
}).sort("coefficient", descending=True)

coefficients


feature,coefficient
str,f64
"""prev_rolling_3_targets""",0.508796
"""prev_season_avg_ppr""",0.432601
"""prev_rolling_3_receptions""",0.28093
"""prev_rolling_3_receiving_yards""",0.020157
"""prev_defense_wr_ppr_allowed_av…",0.015629
"""prev_rolling_3_ppr""",-0.068516


In [19]:
rf_feature_importance = pl.DataFrame({
    "feature": FEATURE_COLUMNS,
    "importance": random_forest.feature_importances_,
}).sort("importance", descending=True)

rf_feature_importance


feature,importance
str,f64
"""prev_season_avg_ppr""",0.521674
"""prev_rolling_3_targets""",0.118991
"""prev_defense_wr_ppr_allowed_av…",0.117753
"""prev_rolling_3_ppr""",0.098848
"""prev_rolling_3_receiving_yards""",0.094956
"""prev_rolling_3_receptions""",0.047778


Opponent defensive strength features were tested using prior season-to-date WR receiving yards allowed and prior season-to-date WR PPR allowed. These features were prediction-safe because they were shifted before joining to player-week rows.

The defensive features produced only a very small change in Linear Regression performance. Linear Regression remained the best model, but the improvement over the baselines is still modest. The coefficients for both defensive features were small, suggesting that player-level usage and production features are still carrying most of the signal.

Single-player prediction using the reusable src/predict.py helpers, so a projection can be generated for a player without re-running this entire notebook.

In [20]:
from src.predict import predict_next_week_ppr

lamb_projection = predict_next_week_ppr(model, dataset, "CeeDee Lamb")
lamb_projection

15.08

Model persistence with src/modeling.py save_model()/load_model(), so the trained model can be reused for prediction without retraining or opening this notebook.

In [21]:
from src.modeling import save_model, load_model

model_path = project_root / "models" / "linear_regression.joblib"
save_model(model, model_path)
loaded_model = load_model(model_path)

loaded_lamb_projection = predict_next_week_ppr(loaded_model, dataset, "CeeDee Lamb")
loaded_lamb_projection

15.08